
<div class="alert alert-block alert-info">  
    
## Import & Connections
</div>

In [1]:
import numpy as np
import pandas as pd
import hashlib
import glob
import time
import os
import re
import matplotlib.pyplot as plt

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_columns', 50)

In [3]:
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [59]:
start_date = '20250130'
end_date = '20250205'
response_type = 'ONLINE_SALES'
pagename = 'PostOrderScreen'
campaign_name = 'Exp2|Urban Company Bangalore' # Exp2_HeroV2_TataNeu
format = 'heroBannerV2'

In [56]:
print(os.getcwd())
local_dataset = '/Users/E2074/local-datasets/customer/ads-monetisation/page-format-experiment/hero-banner'
print(local_dataset)

/Users/E2074/analytics/customer/Ads-Monetisation/page-format-experiment
/Users/E2074/local-datasets/customer/ads-monetisation/page-format-experiment/hero-banner


<div class="alert alert-block alert-warning">  
    
## Datasets & Functions
</div>

In [57]:
start = datetime.strptime(start_date, '%Y%m%d')
end = datetime.strptime(end_date, '%Y%m%d')

# Generate the list of dates
date_range = [(start + timedelta(days=i)).strftime('%Y%m%d') for i in range((end - start).days + 1)]
print(date_range)

['20250202', '20250203', '20250204', '20250205']


In [58]:
def get_customer_metadata(date_range):

    all_df = []
    
    for date in date_range:
    
        customer_metadata = f"""
        
            with response as (
    
                select
                    distinct
                    yyyymmdd,
                    city,
                    os,
                    json_extract_scalar(metadata1, '$.campaignName') AS campaignName,
                    json_extract_scalar(metadata1, '$.slotUnitId') AS slotUnitId,
                    userid,
                    service,
                    orderstatus,
                    -- totalcardcount,
                    -- cardposition,
                    coalesce(aduuid, adid) response_id
                from 
                    canonical.iceberg_log_telemetry_ads_impressions_immutable
                where 
                    yyyymmdd = '{date}'
                    and os = 'android'
                    and responsetype = '{response_type}'
                    and eventname = 'Ad_Response'
                    and pagename = '{pagename}'
                    and format = '{format}'
                    and regexp_like(metadata1, '{campaign_name}')
                ),
                
                impression as (
                
                select
                    distinct
                    yyyymmdd,
                    userid,
                    coalesce(aduuid, adid) impression_id
                from 
                    canonical.iceberg_log_telemetry_ads_impressions_immutable
                where 
                    yyyymmdd = '{date}'
                    and os = 'android'
                    and responsetype = '{response_type}'
                    and eventname = 'Ad_Impression'
                    and pagename = '{pagename}'
                    and format = '{format}'
                    and regexp_like(metadata1, '{campaign_name}')
                ),
                
                viewable_impression as (
                
                select
                    distinct
                    yyyymmdd,
                    userid,
                    coalesce(aduuid, adid) viewable_impression_id
                from 
                    canonical.iceberg_log_telemetry_ads_impressions_immutable
                where 
                    yyyymmdd = '{date}'
                    and os = 'android'
                    and responsetype = '{response_type}'
                    and eventname = 'Ad_Viewable_Impression'
                    and pagename = '{pagename}'
                    and format = '{format}'
                    and regexp_like(metadata1, '{campaign_name}')
                ),
                
                click as (
                
                select
                    distinct
                    yyyymmdd,
                    userid,
                    coalesce(aduuid, adid) click_id
                from 
                    canonical.iceberg_log_telemetry_ads_impressions_immutable
                where 
                    yyyymmdd = '{date}'
                    and os = 'android'
                    and responsetype = '{response_type}'
                    and eventname = 'Ad_Click'
                    and pagename = '{pagename}'
                    and format = '{format}'
                )
                
                select
                    a.*,
                    impression_id,
                    coalesce(click_id, viewable_impression_id) viewable_impression_id,
                    click_id
                from 
                    response a
                left join 
                    impression b
                    on a.yyyymmdd = b.yyyymmdd
                    and a.userid = b.userid
                    and a.response_id = b.impression_id
                left join 
                    viewable_impression c
                    on b.yyyymmdd = c.yyyymmdd
                    and b.userid = c.userid
                    and b.impression_id = c.viewable_impression_id
                left join 
                    click d
                    on b.yyyymmdd = d.yyyymmdd
                    and b.userid = d.userid
                    and b.impression_id = d.click_id
        """
        
        df = pd.read_sql(customer_metadata, connection)
        all_df.append(df)
        df.to_parquet(local_dataset + '/data_dump/Exp3_{}_{}.parquet'.format(campaign_name,date), index=False)
    
    # Concatenate all DataFrames
    final_df = pd.concat(all_df, ignore_index=True)

    return final_df


# df_data = get_customer_metadata(date_range)

In [61]:
def get_customer_segment():    
    
    customer_segment = f"""
    
        with response as (

            select
                distinct
                userid
            from 
                canonical.iceberg_log_telemetry_ads_impressions_immutable
            where 
                yyyymmdd >= '{start_date}'
                and yyyymmdd <= '{end_date}'
                and os = 'android'
                and responsetype = '{response_type}'
                and eventname = 'Ad_Response'
                and pagename = '{pagename}'
                and format = '{format}'
                and regexp_like(metadata1, '{campaign_name}')
            ),

            segment as (
    
            select
                customer_id,
                gender_tag,
                taxi_ltr_segment,
                taxi_retention_segments,
                taxi_regularity_segment,
                taxi_fe_regularity_segment,
                taxi_income_segment,
                taxi_service_affinity,
                taxi_gc_tag
            from 
                datasets.iallocator_customer_segments
            where 
                run_date = '2025-01-29'
            )
            
            select
                userid,
                coalesce(gender_tag, 'UNKNOWN') gender,
                coalesce(taxi_ltr_segment, 'NEW') ltr_segment,
                coalesce(taxi_retention_segments, 'UNKNOWN') retention_segments,
                coalesce(taxi_regularity_segment, 'UNKNOWN') regularity_segment,
                coalesce(taxi_fe_regularity_segment, 'UNKNOWN') fe_regularity_segment,
                coalesce(taxi_income_segment, 'UNKNOWN') income_segment,
                coalesce(taxi_service_affinity, 'UNKNOWN') service_affinity,
                coalesce(taxi_gc_tag, 'UNKNOWN') gc_tag
            from 
                response
            left join 
                segment
                on userid = customer_id
            
    """
    
    df = pd.read_sql(customer_segment, connection)
    df.to_parquet(local_dataset + '/Exp3_customer_segment.parquet', index=False)

    return df

# df_data = get_customer_segment()


<div class="alert alert-block alert-info">  
    
## Load data
</div>

In [63]:
def get_customer_metadata(prefix):
    
    parquet_files = glob.glob(os.path.join(local_dataset + '/data_dump/', f'{prefix}*.parquet'))
    
    customer_metadata = pd.concat([pd.read_parquet(file) for file in parquet_files], ignore_index=True)
    customer_segment = pd.read_parquet(local_dataset + f'/{prefix}_customer_segment.parquet')
    
    df = pd.merge(customer_metadata, customer_segment, how='left', on = 'userid')
    return df

In [64]:
df_exp1 = get_customer_metadata('Exp1')
df_exp2 = get_customer_metadata('Exp2')
df_exp3 = get_customer_metadata('Exp3')

In [65]:
df_exp1['Experiment'] = 'Experiment-1'
df_exp2['Experiment'] = 'Experiment-2'
df_exp3['Experiment'] = 'Experiment-3'

In [66]:
frames = [df_exp1, df_exp2, df_exp3]
df_data = pd.concat(frames)

In [67]:
df_data.head(2)

,yyyymmdd,city,os,campaignName,slotUnitId,userid,service,orderstatus,response_id,impression_id,viewable_impression_id,click_id,gender,ltr_segment,retention_segments,regularity_segment,fe_regularity_segment,income_segment,service_affinity,gc_tag,Experiment
0,20250119,Hyderabad,android,Exp1_HeroV2_TataNeu,PostOrderStarted2,676f87131d37e3d34876a364,Auto,started,6097aab4-4c2c-4293-806c-82b6db31000f676f87131d...,6097aab4-4c2c-4293-806c-82b6db31000f676f87131d...,6097aab4-4c2c-4293-806c-82b6db31000f676f87131d...,None,MALE,HH,HH,UNKNOWN,MONTHLY,LOW_INCOME,UNKNOWN,UNKNOWN,Experiment-1
1,20250119,Bangalore,android,Exp1_HeroV2_TataNeu,PostOrderOnTheWay2,66a5dff7fc10f2126b031cf1,Auto,onTheWay,f753d406-7a63-4071-889d-c80d4ce3b72166a5dff7fc...,f753d406-7a63-4071-889d-c80d4ce3b72166a5dff7fc...,f753d406-7a63-4071-889d-c80d4ce3b72166a5dff7fc...,None,FEMALE,PHH,GOLD,BI_WEEKLY,BI_WEEKLY,MEDIUM_INCOME,UNKNOWN,UNKNOWN,Experiment-1


In [68]:
df_data['yyyymmdd'].value_counts().sort_index(ascending=True)

yyyymmdd
20250117        90
20250118    306764
20250119    308269
20250120    309985
20250121    308642
20250122    303250
20250123    288764
20250124    287889
20250125    295176
20250126    299580
20250127    307895
20250128    304114
20250129    307860
20250130    256182
20250131    556251
20250201    538629
20250202    303467
20250203    456373
20250204    547681
20250205    561682
Name: count, dtype: int64

<div class="alert alert-block alert-warning">  
    
## Funnel Func
</div>

In [70]:

def get_funnel(df, groupby):

    df_temp = df\
                .groupby(groupby)\
                .agg(responses = ('response_id', 'nunique'),
                     impression = ('impression_id', 'nunique'),
                     viewable_impression = ('viewable_impression_id', 'nunique'),
                     click = ('click_id', 'nunique'),
                    )\
                .reset_index()

    df_temp['% render rate'] = (df_temp['impression']*100.00/df_temp['responses']).round(2)
    df_temp['% view'] = (df_temp['viewable_impression']*100.00/df_temp['impression']).round(2)
    df_temp['% ctr'] = (df_temp['click']*100.00/df_temp['viewable_impression']).round(2)

    df_temp['format'] = format
    
    columns = ['format'] + groupby + ['responses', 'impression', 'viewable_impression', 'click', '% render rate', '% view', '% ctr']
    
    return df_temp[columns].sort_values(by=['Experiment', '% ctr'], ascending=[True, False])

<div class="alert alert-block alert-success">  
    
## Analysis
</div>

In [71]:
df_data.head(5)

,yyyymmdd,city,os,campaignName,slotUnitId,userid,service,orderstatus,response_id,impression_id,viewable_impression_id,click_id,gender,ltr_segment,retention_segments,regularity_segment,fe_regularity_segment,income_segment,service_affinity,gc_tag,Experiment
0,20250119,Hyderabad,android,Exp1_HeroV2_TataNeu,PostOrderStarted2,676f87131d37e3d34876a364,Auto,started,6097aab4-4c2c-4293-806c-82b6db31000f676f87131d...,6097aab4-4c2c-4293-806c-82b6db31000f676f87131d...,6097aab4-4c2c-4293-806c-82b6db31000f676f87131d...,None,MALE,HH,HH,UNKNOWN,MONTHLY,LOW_INCOME,UNKNOWN,UNKNOWN,Experiment-1
1,20250119,Bangalore,android,Exp1_HeroV2_TataNeu,PostOrderOnTheWay2,66a5dff7fc10f2126b031cf1,Auto,onTheWay,f753d406-7a63-4071-889d-c80d4ce3b72166a5dff7fc...,f753d406-7a63-4071-889d-c80d4ce3b72166a5dff7fc...,f753d406-7a63-4071-889d-c80d4ce3b72166a5dff7fc...,None,FEMALE,PHH,GOLD,BI_WEEKLY,BI_WEEKLY,MEDIUM_INCOME,UNKNOWN,UNKNOWN,Experiment-1
2,20250119,Bangalore,android,Exp1_HeroV2_TataNeu,PostOrderOnTheWay2,672d535d323c904f9840cfbe,Link,onTheWay,5973be36-c638-412f-ac05-b6515030844f672d535d32...,5973be36-c638-412f-ac05-b6515030844f672d535d32...,5973be36-c638-412f-ac05-b6515030844f672d535d32...,None,MALE,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,MEDIUM_INCOME,UNKNOWN,UNKNOWN,Experiment-1
3,20250119,Hyderabad,android,Exp1_HeroV2_TataNeu,PostOrderStarted2,6208fc428e29c807291bdaf0,Link,started,c7c2e995-5399-426f-a9b6-3035b605d3026208fc428e...,c7c2e995-5399-426f-a9b6-3035b605d3026208fc428e...,c7c2e995-5399-426f-a9b6-3035b605d3026208fc428e...,None,MALE,PHH,SILVER,MONTHLY,MONTHLY,MEDIUM_INCOME,UNKNOWN,UNKNOWN,Experiment-1
4,20250119,Bangalore,android,Exp1_HeroV2_TataNeu,PostOrderStarted2,64e5b8af7284de2fb4552e7d,Auto,started,ff12a600-4f51-4499-b9d9-0546f142aecb64e5b8af72...,ff12a600-4f51-4499-b9d9-0546f142aecb64e5b8af72...,None,None,MALE,PHH,DORMANT,RARE_NEED,UNKNOWN,HIGH_INCOME,UNKNOWN,UNKNOWN,Experiment-1


#### Bivariate data 

In [72]:
get_funnel(df_data, ['Experiment', 'os']).pivot(index = ['Experiment'] , 
                                                columns = ['os'] , 
                                                values =['% render rate', '% view',	'% ctr', 'impression', 'viewable_impression', 'click'])

,% render rate,% view,% ctr,impression,viewable_impression,click
os,android,android,android,android,android,android
Experiment,,,,,,
Experiment-1,80.37,74.23,0.28,1235236.0,916891.0,2568.0
Experiment-2,76.94,75.00,0.04,1609017.0,1206824.0,526.0
Experiment-3,82.23,78.80,0.37,2648106.0,2086748.0,7618.0


In [73]:
get_funnel(df_data, ['Experiment', 'campaignName', 'os']).pivot(index = ['Experiment', 'campaignName'] , 
                                                columns = ['os'] , 
                                                values =['% render rate', '% view',	'% ctr', 'impression', 'viewable_impression', 'click'])

% render rate  % view   % ctr impression  \
os                                         android android android    android   
Experiment   campaignName                                                       
Experiment-1 Exp1_HeroV2_TataNeu             80.37   74.23    0.28  1235236.0   
Experiment-2 Exp2_HeroV2_TataNeu             76.94   75.00    0.04  1609017.0   
Experiment-3 Exp2_HeroV2_TataNeu             82.01   77.83    0.21  1512501.0   
             Urban Company Bangalore         82.53   80.10    0.56  1135607.0   

                                     viewable_impression   click  
os                                               android android  
Experiment   campaignName                                         
Experiment-1 Exp1_HeroV2_TataNeu                916891.0  2568.0  
Experiment-2 Exp2_HeroV2_TataNeu               1206824.0   526.0  
Experiment-3 Exp2_HeroV2_TataNeu               1177124.0  2512.0  
             Urban Company Bangalore            909626.0  5106.0

In [74]:
display(get_funnel(df_data, ['Experiment', 'campaignName', 'yyyymmdd']))

,format,Experiment,campaignName,yyyymmdd,responses,impression,viewable_impression,click,% render rate,% view,% ctr
2,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,20250119,308269,249208,182981,560,80.84,73.43,0.31
3,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,20250120,309985,249442,185623,522,80.47,74.42,0.28
4,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,20250121,308642,246705,182847,504,79.93,74.12,0.28
1,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,20250118,306764,247421,184633,498,80.66,74.62,0.27
5,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,20250122,303250,242379,180758,484,79.93,74.58,0.27
0,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,20250117,90,81,49,0,90.00,60.49,0.00
7,heroBannerV2,Experiment-2,Exp2_HeroV2_TataNeu,20250124,287889,229817,171179,89,79.83,74.48,0.05
9,heroBannerV2,Experiment-2,Exp2_HeroV2_TataNeu,20250126,299580,217592,159638,78,72.63,73.37,0.05
6,heroBannerV2,Experiment-2,Exp2_HeroV2_TataNeu,20250123,288764,230328,171207,76,79.76,74.33,0.04
8,heroBannerV2,Experiment-2,Exp2_HeroV2_TataNeu,20250125,295176,213687,158669,61,72.39,74.25,0.04


In [75]:
display(get_funnel(df_data, ['Experiment', 'campaignName', 'slotUnitId']))

,format,Experiment,campaignName,slotUnitId,responses,impression,viewable_impression,click,% render rate,% view,% ctr
2,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,PostOrderStarted2,377491,297946,237042,1208,78.93,79.56,0.51
1,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,PostOrderOnTheWay2,814829,663489,451745,1011,81.43,68.09,0.22
0,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,PostOrderArrived2,344680,273801,228104,349,79.44,83.31,0.15
5,heroBannerV2,Experiment-2,Exp2_HeroV2_TataNeu,PostOrderStarted3,499504,374509,299872,236,74.98,80.07,0.08
4,heroBannerV2,Experiment-2,Exp2_HeroV2_TataNeu,PostOrderOnTheWay3,1116571,859955,593652,228,77.02,69.03,0.04
3,heroBannerV2,Experiment-2,Exp2_HeroV2_TataNeu,PostOrderArrived3,475203,374553,313300,62,78.82,83.65,0.02
10,heroBannerV2,Experiment-3,Urban Company Bangalore,PostOrderStarted3,480745,388961,342921,2733,80.91,88.16,0.80
9,heroBannerV2,Experiment-3,Urban Company Bangalore,PostOrderOnTheWay3,895227,746646,566705,2373,83.40,75.90,0.42
8,heroBannerV2,Experiment-3,Exp2_HeroV2_TataNeu,PostOrderStarted3,460013,373255,306768,956,81.14,82.19,0.31
7,heroBannerV2,Experiment-3,Exp2_HeroV2_TataNeu,PostOrderOnTheWay3,970699,807803,583076,1235,83.22,72.18,0.21


In [76]:
display(get_funnel(df_data, ['Experiment', 'campaignName', 'service']))

,format,Experiment,campaignName,service,responses,impression,viewable_impression,click,% render rate,% view,% ctr
11,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,CabPremium,8232,6424,5184,52,78.04,80.70,1.00
2,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,Auto NCR,1450,1159,862,5,79.93,74.37,0.58
10,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,CabEconomy,398889,317314,231210,712,79.55,72.86,0.31
9,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,C2C,4097,3221,2325,7,78.62,72.18,0.30
6,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,Bike Lite,40495,33138,24675,72,81.83,74.46,0.29
0,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,Auto,473986,379414,281705,761,80.05,74.25,0.27
13,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,Link,589296,478048,358403,938,81.12,74.97,0.26
7,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,Bike Metro,18804,15116,11610,21,80.39,76.81,0.18
1,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,Auto C2C,173,128,85,0,73.99,66.41,0.00
3,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,Auto Pet,361,277,126,0,76.73,45.49,0.00


In [77]:
display(get_funnel(df_data, ['Experiment', 'campaignName', 'gender']))

,format,Experiment,campaignName,gender,responses,impression,viewable_impression,click,% render rate,% view,% ctr
3,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,UNKNOWN,135736,111869,81189,336,82.42,72.58,0.41
2,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,OTHERS,6246,5231,3597,14,83.75,68.76,0.39
0,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,FEMALE,307611,248961,186104,522,80.93,74.75,0.28
1,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,MALE,1087407,869175,646001,1696,79.93,74.32,0.26
7,heroBannerV2,Experiment-2,Exp2_HeroV2_TataNeu,UNKNOWN,163795,130323,95642,47,79.56,73.39,0.05
4,heroBannerV2,Experiment-2,Exp2_HeroV2_TataNeu,FEMALE,433361,334747,254139,94,77.24,75.92,0.04
5,heroBannerV2,Experiment-2,Exp2_HeroV2_TataNeu,MALE,1485363,1137018,852136,383,76.55,74.94,0.04
6,heroBannerV2,Experiment-2,Exp2_HeroV2_TataNeu,OTHERS,8759,6929,4907,2,79.11,70.82,0.04
15,heroBannerV2,Experiment-3,Urban Company Bangalore,UNKNOWN,77680,65480,51626,399,84.29,78.84,0.77
12,heroBannerV2,Experiment-3,Urban Company Bangalore,FEMALE,508198,420268,334252,2064,82.70,79.53,0.62


In [78]:
display(get_funnel(df_data, ['Experiment', 'campaignName', 'ltr_segment']))

,format,Experiment,campaignName,ltr_segment,responses,impression,viewable_impression,click,% render rate,% view,% ctr
1,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,NEW,106177,88154,63460,286,83.03,71.99,0.45
3,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,UNKNOWN,26853,22306,14806,59,83.07,66.38,0.40
0,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,HH,181661,147854,107697,383,81.39,72.84,0.36
2,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,PHH,1222309,976922,730928,1840,79.92,74.82,0.25
7,heroBannerV2,Experiment-2,Exp2_HeroV2_TataNeu,UNKNOWN,37354,29958,19500,15,80.20,65.09,0.08
4,heroBannerV2,Experiment-2,Exp2_HeroV2_TataNeu,HH,242651,191551,139534,89,78.94,72.84,0.06
5,heroBannerV2,Experiment-2,Exp2_HeroV2_TataNeu,NEW,123517,99547,72346,41,80.59,72.68,0.06
6,heroBannerV2,Experiment-2,Exp2_HeroV2_TataNeu,PHH,1687756,1287961,975444,381,76.31,75.74,0.04
13,heroBannerV2,Experiment-3,Urban Company Bangalore,NEW,45011,38418,30055,274,85.35,78.23,0.91
15,heroBannerV2,Experiment-3,Urban Company Bangalore,UNKNOWN,12249,10409,7304,64,84.98,70.17,0.88


In [79]:
display(get_funnel(df_data, ['Experiment', 'campaignName', 'retention_segments']))

,format,Experiment,campaignName,retention_segments,responses,impression,viewable_impression,click,% render rate,% view,% ctr
8,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,UNKNOWN,133030,110460,78266,345,83.03,70.85,0.44
4,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,INACTIVE,117273,94972,70251,259,80.98,73.97,0.37
3,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,HH,74987,60854,44202,147,81.15,72.64,0.33
0,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,DORMANT,251107,200864,149262,447,79.99,74.31,0.30
7,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,SILVER,119183,94967,71123,205,79.68,74.89,0.29
2,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,GOLD,222855,177961,132976,360,79.86,74.72,0.27
6,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,PRIME,11408,9091,6857,17,79.69,75.43,0.25
5,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,PLATINUM,275544,219749,164667,378,79.75,74.93,0.23
1,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,ELITE,331613,266318,199287,410,80.31,74.83,0.21
12,heroBannerV2,Experiment-2,Exp2_HeroV2_TataNeu,HH,111403,87643,63717,36,78.67,72.70,0.06


In [80]:
display(get_funnel(df_data, ['Experiment', 'campaignName', 'regularity_segment']))

,format,Experiment,campaignName,regularity_segment,responses,impression,viewable_impression,click,% render rate,% view,% ctr
4,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,UNKNOWN,314691,258314,185963,728,82.08,71.99,0.39
3,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,RARE_NEED,165352,132299,98981,314,80.01,74.82,0.32
2,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,MONTHLY,156668,125357,94207,266,80.01,75.15,0.28
0,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,BI_WEEKLY,284728,227132,170173,443,79.77,74.92,0.26
5,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,WEEKLY,343799,273712,205336,496,79.61,75.02,0.24
1,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,DAILY,271762,218422,162231,321,80.37,74.27,0.20
9,heroBannerV2,Experiment-2,Exp2_HeroV2_TataNeu,RARE_NEED,209150,160571,121654,86,76.77,75.76,0.07
10,heroBannerV2,Experiment-2,Exp2_HeroV2_TataNeu,UNKNOWN,403522,321056,231380,145,79.56,72.07,0.06
6,heroBannerV2,Experiment-2,Exp2_HeroV2_TataNeu,BI_WEEKLY,384758,293843,222598,91,76.37,75.75,0.04
8,heroBannerV2,Experiment-2,Exp2_HeroV2_TataNeu,MONTHLY,202239,154842,117002,52,76.56,75.56,0.04


In [81]:
display(get_funnel(df_data, ['Experiment', 'campaignName', 'fe_regularity_segment']))

,format,Experiment,campaignName,fe_regularity_segment,responses,impression,viewable_impression,click,% render rate,% view,% ctr
4,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,UNKNOWN,340366,277050,203632,767,81.40,73.50,0.38
3,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,RARE_NEED,82809,66926,49015,156,80.82,73.24,0.32
2,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,MONTHLY,222492,178440,132604,398,80.20,74.31,0.30
0,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,BI_WEEKLY,365754,292048,217862,573,79.85,74.60,0.26
5,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,WEEKLY,226766,180782,135698,311,79.72,75.06,0.23
1,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,DAILY,298813,239990,178080,363,80.31,74.20,0.20
10,heroBannerV2,Experiment-2,Exp2_HeroV2_TataNeu,UNKNOWN,417577,328412,243326,172,78.65,74.09,0.07
9,heroBannerV2,Experiment-2,Exp2_HeroV2_TataNeu,RARE_NEED,115927,90175,66518,37,77.79,73.77,0.06
8,heroBannerV2,Experiment-2,Exp2_HeroV2_TataNeu,MONTHLY,297169,228458,170526,78,76.88,74.64,0.05
6,heroBannerV2,Experiment-2,Exp2_HeroV2_TataNeu,BI_WEEKLY,495514,378643,285964,107,76.41,75.52,0.04


In [82]:
display(get_funnel(df_data, ['Experiment', 'campaignName', 'income_segment']))

,format,Experiment,campaignName,income_segment,responses,impression,viewable_impression,click,% render rate,% view,% ctr
3,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,UNKNOWN,399929,319864,234221,809,79.98,73.23,0.35
2,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,MEDIUM_INCOME,481159,409883,310877,833,85.19,75.85,0.27
1,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,LOW_INCOME,81433,70527,53452,138,86.61,75.79,0.26
0,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,HIGH_INCOME,574479,434962,318341,788,75.71,73.19,0.25
7,heroBannerV2,Experiment-2,Exp2_HeroV2_TataNeu,UNKNOWN,517611,398078,295021,163,76.91,74.11,0.06
4,heroBannerV2,Experiment-2,Exp2_HeroV2_TataNeu,HIGH_INCOME,783527,564398,416591,170,72.03,73.81,0.04
5,heroBannerV2,Experiment-2,Exp2_HeroV2_TataNeu,LOW_INCOME,117432,97458,74111,27,82.99,76.04,0.04
6,heroBannerV2,Experiment-2,Exp2_HeroV2_TataNeu,MEDIUM_INCOME,672708,549083,421101,166,81.62,76.69,0.04
15,heroBannerV2,Experiment-3,Urban Company Bangalore,UNKNOWN,281741,229233,182771,1212,81.36,79.73,0.66
14,heroBannerV2,Experiment-3,Urban Company Bangalore,MEDIUM_INCOME,410207,361920,292288,1778,88.23,80.76,0.61


In [83]:
display(get_funnel(df_data, ['Experiment', 'campaignName', 'service_affinity']))

,format,Experiment,campaignName,service_affinity,responses,impression,viewable_impression,click,% render rate,% view,% ctr
7,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,UNKNOWN,831450,669981,495667,1647,80.58,73.98,0.33
1,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,LINK_AUTO,40703,32898,24933,71,80.82,75.79,0.28
4,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,ONLY_AUTO,187133,148500,110964,287,79.36,74.72,0.26
0,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,AUTO_CAB,17062,13216,9738,21,77.46,73.68,0.22
2,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,LINK_CAB,10504,8557,6190,13,81.46,72.34,0.21
3,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,NO_AFFINITY,93241,74078,54367,116,79.45,73.39,0.21
5,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,ONLY_CAB,50931,39807,28737,61,78.16,72.19,0.21
6,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,ONLY_LINK,305976,248199,186295,352,81.12,75.06,0.19
10,heroBannerV2,Experiment-2,Exp2_HeroV2_TataNeu,LINK_CAB,14748,11684,8440,5,79.22,72.24,0.06
15,heroBannerV2,Experiment-2,Exp2_HeroV2_TataNeu,UNKNOWN,1069839,830306,618696,346,77.61,74.51,0.06


In [84]:
display(get_funnel(df_data, ['Experiment', 'campaignName', 'gc_tag']))

,format,Experiment,campaignName,gc_tag,responses,impression,viewable_impression,click,% render rate,% view,% ctr
2,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,UNKNOWN,1226827,989524,733136,2146,80.66,74.09,0.29
1,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,NON_GC,285660,226145,168919,395,79.17,74.69,0.23
0,heroBannerV2,Experiment-1,Exp1_HeroV2_TataNeu,GC,24513,19567,14836,27,79.82,75.82,0.18
4,heroBannerV2,Experiment-2,Exp2_HeroV2_TataNeu,NON_GC,387649,293408,221812,87,75.69,75.60,0.04
5,heroBannerV2,Experiment-2,Exp2_HeroV2_TataNeu,UNKNOWN,1668105,1288163,963980,433,77.22,74.83,0.04
3,heroBannerV2,Experiment-2,Exp2_HeroV2_TataNeu,GC,35524,27446,21032,6,77.26,76.63,0.03
11,heroBannerV2,Experiment-3,Urban Company Bangalore,UNKNOWN,1126103,931967,744718,4304,82.76,79.91,0.58
9,heroBannerV2,Experiment-3,Urban Company Bangalore,GC,25456,20785,16741,90,81.65,80.54,0.54
10,heroBannerV2,Experiment-3,Urban Company Bangalore,NON_GC,224413,182855,148167,712,81.48,81.03,0.48
6,heroBannerV2,Experiment-3,Exp2_HeroV2_TataNeu,GC,30856,25280,20154,56,81.93,79.72,0.28


#### Multivariate data 

In [85]:
get_funnel(df_data, ['Experiment', 'slotUnitId', 'campaignName', 'os']).pivot(index = ['slotUnitId', 'Experiment', 'campaignName'] , 
                                                columns = ['os'] , 
                                                values =['% render rate', '% view',	'% ctr'])

% render rate  % view  \
os                                                            android android   
slotUnitId         Experiment   campaignName                                    
PostOrderArrived2  Experiment-1 Exp1_HeroV2_TataNeu             79.44   83.31   
PostOrderArrived3  Experiment-2 Exp2_HeroV2_TataNeu             78.82   83.65   
                   Experiment-3 Exp2_HeroV2_TataNeu             80.14   86.68   
PostOrderOnTheWay2 Experiment-1 Exp1_HeroV2_TataNeu             81.43   68.09   
PostOrderOnTheWay3 Experiment-2 Exp2_HeroV2_TataNeu             77.02   69.03   
                   Experiment-3 Exp2_HeroV2_TataNeu             83.22   72.18   
                                Urban Company Bangalore         83.40   75.90   
PostOrderStarted2  Experiment-1 Exp1_HeroV2_TataNeu             78.93   79.56   
PostOrderStarted3  Experiment-2 Exp2_HeroV2_TataNeu             74.98   80.07   
                   Experiment-3 Exp2_HeroV2_TataNeu             81.14   82.19   
                                Urban Company Bangalore         80.91   88.16   

                                                          % ctr  
os                                                      android  
slotUnitId         Experiment   campaignName                     
PostOrderArrived2  Experiment-1 Exp1_HeroV2_TataNeu        0.15  
PostOrderArrived3  Experiment-2 Exp2_HeroV2_TataNeu        0.02  
                   Experiment-3 Exp2_HeroV2_TataNeu        0.11  
PostOrderOnTheWay2 Experiment-1 Exp1_HeroV2_TataNeu        0.22  
PostOrderOnTheWay3 Experiment-2 Exp2_HeroV2_TataNeu        0.04  
                   Experiment-3 Exp2_HeroV2_TataNeu        0.21  
                                Urban Company Bangalore    0.42  
PostOrderStarted2  Experiment-1 Exp1_HeroV2_TataNeu        0.51  
PostOrderStarted3  Experiment-2 Exp2_HeroV2_TataNeu        0.08  
                   Experiment-3 Exp2_HeroV2_TataNeu        0.31  
                                Urban Company Bangalore    0.80

In [86]:
display(get_funnel(df_data, ['Experiment', 'yyyymmdd', 'campaignName', 'slotUnitId']))

,format,Experiment,yyyymmdd,campaignName,slotUnitId,responses,impression,viewable_impression,click,% render rate,% view,% ctr
11,heroBannerV2,Experiment-1,20250120,Exp1_HeroV2_TataNeu,PostOrderStarted2,76107,59952,48204,255,78.77,80.40,0.53
14,heroBannerV2,Experiment-1,20250121,Exp1_HeroV2_TataNeu,PostOrderStarted2,75996,59921,47442,251,78.85,79.17,0.53
8,heroBannerV2,Experiment-1,20250119,Exp1_HeroV2_TataNeu,PostOrderStarted2,74963,59392,47102,239,79.23,79.31,0.51
17,heroBannerV2,Experiment-1,20250122,Exp1_HeroV2_TataNeu,PostOrderStarted2,74745,58754,46554,234,78.61,79.24,0.50
5,heroBannerV2,Experiment-1,20250118,Exp1_HeroV2_TataNeu,PostOrderStarted2,75663,59910,47727,229,79.18,79.66,0.48
7,heroBannerV2,Experiment-1,20250119,Exp1_HeroV2_TataNeu,PostOrderOnTheWay2,165695,135676,90853,245,81.88,66.96,0.27
4,heroBannerV2,Experiment-1,20250118,Exp1_HeroV2_TataNeu,PostOrderOnTheWay2,162179,132496,90925,197,81.70,68.62,0.22
10,heroBannerV2,Experiment-1,20250120,Exp1_HeroV2_TataNeu,PostOrderOnTheWay2,164947,134768,91716,197,81.70,68.05,0.21
13,heroBannerV2,Experiment-1,20250121,Exp1_HeroV2_TataNeu,PostOrderOnTheWay2,162654,131467,89499,189,80.83,68.08,0.21
16,heroBannerV2,Experiment-1,20250122,Exp1_HeroV2_TataNeu,PostOrderOnTheWay2,159296,129032,88726,183,81.00,68.76,0.21
